# LeetCode 1089: Duplicate Zeros

**Difficulty**: Easy  
**Topics**: Array, Two Pointers  
**Link**: [LeetCode Problem](https://leetcode.com/problems/duplicate-zeros/)

---

## Problem Statement

Given a fixed-length integer array `arr`, duplicate each occurrence of zero, shifting the remaining elements to the right.

**Note**: The elements beyond the length of the original array are not written. Do the above modifications to the input array **in place** and do not return anything.

### Visual Example

```
Input:  [1, 0, 2, 3, 0, 4, 5, 0]
           ↓     ↓        ↓
        Duplicate these zeros

Step 1: [1, 0, 0, 2, 3, 0, 4, 5]  (first 0 duplicated, shift right)
Step 2: [1, 0, 0, 2, 3, 0, 0, 4]  (second 0 duplicated, shift right)
Step 3: [1, 0, 0, 2, 3, 0, 0, 4]  (third 0 would duplicate but 5 falls off)

Output: [1, 0, 0, 2, 3, 0, 0, 4]
```

### Examples

**Example 1:**
```
Input: arr = [1,0,2,3,0,4,5,0]
Output: [1,0,0,2,3,0,0,4]
Explanation: After calling your function, the input array is modified to: [1,0,0,2,3,0,0,4]
```

**Example 2:**
```
Input: arr = [1,2,3]
Output: [1,2,3]
Explanation: There are no zeros, so the array remains unchanged.
```

### Constraints

- `1 <= arr.length <= 10^4`
- `0 <= arr[i] <= 9`

---

## Approach 1: Using Extra Space (Not In-Place)

### Intuition

The simplest approach is to create a new array and copy elements, duplicating zeros as we go.

### Algorithm

```
1. Create a new array
2. For each element in original array:
     If element is 0:
         Add two zeros to new array (if space)
     Else:
         Add element to new array
3. Copy new array back to original
```

### Complexity

- **Time**: O(n)
- **Space**: O(n) - uses extra array

**Note**: This violates the in-place requirement, but helps understand the problem.

In [ ]:
def duplicateZeros_extraSpace(arr):
    """
    Using extra space (not in-place).
    Time: O(n), Space: O(n)
    """
    n = len(arr)
    result = []
    
    for num in arr:
        if len(result) >= n:
            break
        
        if num == 0:
            result.append(0)
            if len(result) < n:
                result.append(0)
        else:
            result.append(num)
    
    # Copy back to original array
    for i in range(n):
        arr[i] = result[i]

# Test
test_cases = [
    [1, 0, 2, 3, 0, 4, 5, 0],
    [1, 2, 3],
    [0, 0, 0, 0],
]

print("Extra Space Approach:\n")
for arr in test_cases:
    original = arr.copy()
    duplicateZeros_extraSpace(arr)
    print(f"Input:  {original}")
    print(f"Output: {arr}\n")

---

## Approach 2: Two-Pass In-Place (Optimal)

### Intuition

To solve this in-place without extra space, we need a clever approach:

**Key Insight**: Work **backwards** to avoid overwriting elements we haven't processed yet.

**Two-Pass Strategy**:
1. **Pass 1 (Left to Right)**: Count how many zeros will be duplicated
2. **Pass 2 (Right to Left)**: Place elements in their final positions

### Why Work Backwards?

If we work forwards, duplicating a zero would overwrite the next element before we process it.

Working backwards:
- We know the final position of each element
- We can safely place elements without overwriting unprocessed data

### Visual Walkthrough

For `[1, 0, 2, 3, 0, 4, 5, 0]`:

**Pass 1: Count zeros to duplicate**
```
Original: [1, 0, 2, 3, 0, 4, 5, 0]
           0  1  2  3  4  5  6  7  (indices)

Simulate filling:
- i=0: 1 → write_pos=0
- i=1: 0 → write_pos=1,2 (duplicate)
- i=2: 2 → write_pos=3
- i=3: 3 → write_pos=4
- i=4: 0 → write_pos=5,6 (duplicate)
- i=5: 4 → write_pos=7
- i=6: 5 → write_pos=8 (out of bounds!)

Stop at i=5 (last element that fits)
```

**Pass 2: Fill backwards**
```
Start from i=5, write_pos=7

i=5: arr[5]=4 → arr[7]=4
[1, 0, 2, 3, 0, 4, 5, 4]

i=4: arr[4]=0 → arr[6]=0, arr[5]=0
[1, 0, 2, 3, 0, 0, 0, 4]

i=3: arr[3]=3 → arr[4]=3
[1, 0, 2, 3, 3, 0, 0, 4]

i=2: arr[2]=2 → arr[3]=2
[1, 0, 2, 2, 3, 0, 0, 4]

i=1: arr[1]=0 → arr[2]=0, arr[1]=0
[1, 0, 0, 2, 3, 0, 0, 4]

i=0: arr[0]=1 → arr[0]=1
[1, 0, 0, 2, 3, 0, 0, 4] ✅
```

### Algorithm

```
Pass 1: Find the last element that will fit
1. possible_dups = 0
2. For i from 0 to n-1:
     If i + possible_dups >= n:
         break (reached the end)
     If arr[i] == 0:
         possible_dups++

Pass 2: Fill array backwards
3. last = n - 1 - possible_dups
4. For i from last down to 0:
     If arr[i] == 0:
         arr[i + possible_dups] = 0
         possible_dups--
         arr[i + possible_dups] = 0
     Else:
         arr[i + possible_dups] = arr[i]
```

### Complexity

- **Time**: O(n) - two passes through the array
- **Space**: O(1) - in-place modification

In [ ]:
def duplicateZeros(arr):
    """
    Optimal in-place two-pass solution.
    Time: O(n), Space: O(1)
    """
    n = len(arr)
    possible_dups = 0
    
    # Pass 1: Count zeros that will be duplicated
    for i in range(n):
        # Stop when we reach the boundary
        if i + possible_dups >= n:
            break
        
        if arr[i] == 0:
            # Edge case: if this zero would be duplicated but only one slot left
            if i + possible_dups == n - 1:
                arr[n - 1] = 0
                n -= 1
                break
            possible_dups += 1
    
    # Pass 2: Fill array backwards
    last = n - 1 - possible_dups
    
    for i in range(last, -1, -1):
        if arr[i] == 0:
            # Duplicate the zero
            arr[i + possible_dups] = 0
            possible_dups -= 1
            arr[i + possible_dups] = 0
        else:
            # Just move the element
            arr[i + possible_dups] = arr[i]

# Test
print("Two-Pass In-Place Approach (Optimal):\n")
for arr in test_cases:
    original = arr.copy()
    duplicateZeros(arr)
    print(f"Input:  {original}")
    print(f"Output: {arr}\n")

### Detailed Step-by-Step Trace

Let's trace through `[1, 0, 2, 3, 0, 4, 5, 0]` with detailed output:

In [ ]:
def duplicateZeros_verbose(arr):
    """Verbose version showing each step."""
    n = len(arr)
    possible_dups = 0
    
    print(f"Input array: {arr}")
    print(f"Length: {n}\n")
    
    print("=" * 60)
    print("PASS 1: Count zeros to duplicate")
    print("=" * 60)
    
    for i in range(n):
        print(f"\ni={i}: arr[{i}]={arr[i]}, possible_dups={possible_dups}")
        print(f"  Check: i + possible_dups = {i} + {possible_dups} = {i + possible_dups}")
        
        if i + possible_dups >= n:
            print(f"  {i + possible_dups} >= {n}, stop here")
            break
        
        if arr[i] == 0:
            if i + possible_dups == n - 1:
                print(f"  Edge case: zero at boundary, set arr[{n-1}]=0")
                arr[n - 1] = 0
                n -= 1
                break
            possible_dups += 1
            print(f"  Found zero! possible_dups → {possible_dups}")
        else:
            print(f"  Not a zero, continue")
    
    print(f"\nPass 1 complete: possible_dups = {possible_dups}")
    
    last = n - 1 - possible_dups
    print(f"Last element to process: last = {n-1} - {possible_dups} = {last}")
    
    print("\n" + "=" * 60)
    print("PASS 2: Fill array backwards")
    print("=" * 60)
    
    for i in range(last, -1, -1):
        print(f"\ni={i}: arr[{i}]={arr[i]}, possible_dups={possible_dups}")
        
        if arr[i] == 0:
            write_pos1 = i + possible_dups
            print(f"  Zero found! Duplicate it:")
            print(f"    arr[{write_pos1}] = 0")
            arr[write_pos1] = 0
            
            possible_dups -= 1
            write_pos2 = i + possible_dups
            print(f"    possible_dups → {possible_dups}")
            print(f"    arr[{write_pos2}] = 0")
            arr[write_pos2] = 0
        else:
            write_pos = i + possible_dups
            print(f"  Move element: arr[{write_pos}] = {arr[i]}")
            arr[write_pos] = arr[i]
        
        print(f"  Array now: {arr}")
    
    print(f"\n{'=' * 60}")
    print(f"Final result: {arr}")
    print("=" * 60)

# Test
test_arr = [1, 0, 2, 3, 0, 4, 5, 0]
duplicateZeros_verbose(test_arr)

---

## Edge Cases

In [ ]:
# Edge case 1: No zeros
print("No zeros:")
arr = [1, 2, 3, 4, 5]
original = arr.copy()
duplicateZeros(arr)
print(f"Input:  {original}")
print(f"Output: {arr}")
print(f"Expected: {original}\n")

# Edge case 2: All zeros
print("All zeros:")
arr = [0, 0, 0, 0]
original = arr.copy()
duplicateZeros(arr)
print(f"Input:  {original}")
print(f"Output: {arr}")
print(f"Expected: [0, 0, 0, 0]\n")

# Edge case 3: Single element (zero)
print("Single zero:")
arr = [0]
original = arr.copy()
duplicateZeros(arr)
print(f"Input:  {original}")
print(f"Output: {arr}")
print(f"Expected: [0]\n")

# Edge case 4: Single element (non-zero)
print("Single non-zero:")
arr = [5]
original = arr.copy()
duplicateZeros(arr)
print(f"Input:  {original}")
print(f"Output: {arr}")
print(f"Expected: [5]\n")

# Edge case 5: Zero at the end (boundary case)
print("Zero at boundary:")
arr = [8, 4, 5, 0, 0, 0, 0, 7]
original = arr.copy()
duplicateZeros(arr)
print(f"Input:  {original}")
print(f"Output: {arr}")
print(f"Expected: [8, 4, 5, 0, 0, 0, 0, 0]\n")

# Edge case 6: Alternating zeros and non-zeros
print("Alternating:")
arr = [1, 0, 2, 0, 3, 0]
original = arr.copy()
duplicateZeros(arr)
print(f"Input:  {original}")
print(f"Output: {arr}")
print(f"Expected: [1, 0, 0, 2, 0, 0]")

---

## Common Mistakes

### Mistake 1: Working Forward (Overwrites Data)

**Wrong**:
```python
# This doesn't work!
i = 0
while i < len(arr):
    if arr[i] == 0:
        # Insert 0 at i+1, shift everything right
        arr.insert(i + 1, 0)
        arr.pop()  # Remove last element
        i += 2  # Skip the duplicated zero
    else:
        i += 1
```

**Problem**: `insert()` is O(n) for each zero, making total time O(n²).

**Right**: Use the two-pass backward approach (O(n) total).

### Mistake 2: Not Handling Boundary Zero

**Wrong**:
```python
# Missing edge case check
if arr[i] == 0:
    possible_dups += 1  # What if this zero is at the boundary?
```

**Right**:
```python
if arr[i] == 0:
    # Check if this zero would be duplicated at the exact boundary
    if i + possible_dups == n - 1:
        arr[n - 1] = 0
        n -= 1
        break
    possible_dups += 1
```

### Mistake 3: Wrong Loop Direction in Pass 2

**Wrong**:
```python
# Going forward overwrites unprocessed elements
for i in range(0, last + 1):
    if arr[i] == 0:
        arr[i + possible_dups] = 0  # Overwrites data!
```

**Right**:
```python
# Go backwards to avoid overwriting
for i in range(last, -1, -1):
    if arr[i] == 0:
        arr[i + possible_dups] = 0  # Safe!
```

### Mistake 4: Forgetting to Decrement possible_dups

**Wrong**:
```python
if arr[i] == 0:
    arr[i + possible_dups] = 0
    arr[i + possible_dups] = 0  # Wrong! Same position
```

**Right**:
```python
if arr[i] == 0:
    arr[i + possible_dups] = 0
    possible_dups -= 1  # ✅ Decrement first
    arr[i + possible_dups] = 0
```

---

## Why This Problem is Tricky

### The Challenge

1. **In-place requirement**: Can't use extra array
2. **Fixed length**: Elements fall off the end
3. **Shifting**: Duplicating zeros shifts everything right
4. **Overwriting risk**: Working forward overwrites unprocessed data

### The Solution Strategy

**Two-pass approach**:
1. **First pass**: Figure out which elements will fit
2. **Second pass**: Work backwards to place elements safely

**Why backwards?**
- We know the final position of each element
- We won't overwrite data we haven't processed yet
- We can duplicate zeros without losing information

### Visual Intuition

Think of it like rearranging books on a shelf:
- **Forward**: You'd need to move books multiple times (inefficient)
- **Backward**: Start from the end, place each book once (efficient)

```
Forward (wrong):
[1, 0, 2, 3] → duplicate 0
[1, 0, ?, 3] → lost 2!

Backward (right):
[1, 0, 2, 3]
      ↓
[1, 0, 2, 3] → place 3 at position 3
   ↓
[1, 0, 0, 2] → place 2 at position 3, duplicate 0
↓
[1, 0, 0, 2] → place 1 at position 0
```

---

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity | In-Place? | Notes |
|----------|----------------|------------------|-----------|-------|
| Extra Array | O(n) | O(n) | ❌ No | Simple but violates requirements |
| Forward with Insert | O(n²) | O(1) | ✅ Yes | Too slow, insert is O(n) |
| Two-Pass Backward | O(n) | O(1) | ✅ Yes | Optimal solution |

The two-pass backward approach is the only solution that meets all requirements:
- O(n) time
- O(1) space
- In-place modification

---

## Related Problems

- [27. Remove Element](https://leetcode.com/problems/remove-element/) - In-place array modification
- [26. Remove Duplicates from Sorted Array](https://leetcode.com/problems/remove-duplicates-from-sorted-array/) - Two pointers in-place
- [283. Move Zeroes](https://leetcode.com/problems/move-zeroes/) - Moving zeros (opposite direction)
- [88. Merge Sorted Array](https://leetcode.com/problems/merge-sorted-array/) - Backward filling pattern
- [80. Remove Duplicates from Sorted Array II](https://leetcode.com/problems/remove-duplicates-from-sorted-array-ii/) - In-place with duplicates

---

## Key Takeaways

1. **In-place array modification** often requires working backwards

2. **Two-pass strategy**:
   - Pass 1: Gather information (count, find boundaries)
   - Pass 2: Modify array based on information

3. **Backward filling** prevents overwriting unprocessed data

4. **Boundary cases** need special handling:
   - Zero at exact boundary position
   - Single element arrays
   - All zeros or no zeros

5. **Fixed-length constraint** means elements can fall off the end

6. **Space optimization**: O(1) space requires clever pointer manipulation

7. **Direction matters**: Forward vs backward can make or break the solution

8. **Edge case**: When a zero would be duplicated but only one slot remains

### The Pattern

**In-place array modification with shifting**:
- Count/calculate first (determine final positions)
- Work backwards (avoid overwriting)
- Handle boundary cases carefully
- Use two pointers (read position and write position)

This pattern appears in many array problems requiring O(1) space!